In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [46]:
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.sample(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
29,8773-HHUOZ,Female,0,No,Yes,17,Yes,No,DSL,No,...,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,64.70,1093.1,Yes
4988,1439-LCGVL,Female,0,Yes,No,26,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.20,459.6,No
4998,8920-NAVAY,Male,1,No,No,62,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,One year,Yes,Bank transfer (automatic),110.75,7053.35,No
356,3351-NGXYI,Female,1,No,No,16,Yes,Yes,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,54.10,889,No
4681,9965-YOKZB,Male,1,No,No,9,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,83.85,790.15,Yes
6208,0909-SELIE,Male,0,Yes,No,61,Yes,Yes,DSL,Yes,...,Yes,Yes,No,Yes,Two year,Yes,Bank transfer (automatic),80.80,4860.85,No
4496,9489-JMTTN,Female,0,Yes,Yes,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),89.75,6595.9,No
5683,5057-LCOUI,Female,0,No,No,39,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,50.75,2011.4,Yes
4710,7816-VGHTO,Female,0,Yes,Yes,6,No,No phone service,DSL,No,...,Yes,Yes,No,No,Two year,No,Mailed check,40.55,217.5,No
61,1215-FIGMP,Male,0,No,No,60,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),89.90,5450.7,No


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [48]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [49]:
df.drop("customerID", axis=1, inplace=True)

In [50]:
df["TotalCharges"].dtypes

dtype('O')

In [ ]:
# it is numeric bt dataset shows as object . so convert it into numeric

df["TotalCharges"] = pd.to_numeric(
    
    df["TotalCharges"],

    errors="coerce"
)

In [52]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Churn"] = le.fit_transform(df["Churn"])

In [53]:
X=df.drop("Churn",axis=1)
y=df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [54]:
# FEATURE ENGINEERING

num_col=X.select_dtypes(include=np.number).columns
cat_col=X.select_dtypes(exclude=np.number).columns
print(num_col)
print(cat_col)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [55]:
#feature engineering
from sklearn.pipeline import Pipeline


num_pipeline=Pipeline(
      steps=[
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
      ]
)

cat_pipeline=Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder())
    ]
)
num_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [56]:
cat_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder())])

In [57]:
prepocessing=ColumnTransformer(
  transformers=[
      ("num", num_pipeline, num_col),
      ("cat", cat_pipeline, cat_col)
  ]
)

prepocessing



ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder', OneHotEncoder())]),
                                 Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])

In [58]:
# modal implement
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
lr=Pipeline(
    steps=[
        ("prepocessing", prepocessing),
        ("model", LogisticRegression())
    ]
)

lr.fit(X_train, y_train)


#Train acuracy
y_pred_train=lr.predict(X_train)
print("logistic regresssion train \n")

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)

print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

#Test acuracy


y_pred_test=lr.predict(X_test)

print("\nlogistic regresssion test \n")

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)
f1=f1_score(y_test,y_pred_test)

print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


logistic regresssion train 

accuracy 0.8059992900248492
pecision 0.6615755627009646
recall 0.5505016722408027
f1 0.6009492515516612

logistic regresssion test 

accuracy 0.8055358410220014
pecision 0.6572327044025157
recall 0.5588235294117647
f1 0.6040462427745664


In [59]:
# random forest hyperperameter tunning


from sklearn.ensemble import RandomForestClassifier

random_forest=Pipeline(
    steps=[
        ("prepocessing", prepocessing),
        ("model", RandomForestClassifier(
               n_estimators=300,
               max_depth=10,
               min_samples_split=5,
               min_samples_leaf=2,
               random_state=42))
    ]
)

random_forest.fit(X_train, y_train)

# train part
y_pred_train=random_forest.predict(X_train)

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)
print("Random Forest train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

# test accauracy
y_pred_test=random_forest.predict(X_test)

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)

f1=f1_score(y_test,y_pred_test)
print("\nRandom Forest test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


# here without hyperperameter random forest also overfitted result shows ... n_estimators=300,max_depth=10, min_samples_split=5,min_samples_leaf=2,random_state=42
# this perameter handles the overfitting condition


Random Forest train part analysis

accuracy 0.8601348952786653
pecision 0.7895167895167895
recall 0.6448160535117057
f1 0.7098674521354934

Random Forest test part analysis

accuracy 0.8055358410220014
pecision 0.6700680272108843
recall 0.5267379679144385
f1 0.5898203592814372


In [60]:

# decision tree classifier
from sklearn.tree import DecisionTreeClassifier

dt=Pipeline(
    steps=[
        ("prepocessing",prepocessing),
        ("model",DecisionTreeClassifier(random_state=42))

    ]

)

dt.fit(X_train, y_train)
# train part
y_pred_train=dt.predict(X_train)

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)
print("Decsion Tree Classifier train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

# test accauracy
y_pred_test=dt.predict(X_test)

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)

f1=f1_score(y_test,y_pred_test)
print("\nDecsion Tree Classifier test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


# heavy overfitting


Decsion Tree Classifier train part analysis

accuracy 0.9980475683351083
pecision 0.9993270524899058
recall 0.9933110367892977
f1 0.996309963099631

Decsion Tree Classifier test part analysis

accuracy 0.7288857345635202
pecision 0.4896373056994819
recall 0.5053475935828877
f1 0.49736842105263157


In [61]:

# decision tree classifier with  perameter tunning

from sklearn.tree import DecisionTreeClassifier

dt=Pipeline(
    steps=[
        ("prepocessing",prepocessing),
        ("model",DecisionTreeClassifier(
                         max_depth=5,
                         min_samples_split=10,
                         min_samples_leaf=5,
                         random_state=42))

    ]

)

dt.fit(X_train, y_train)
# train part
y_pred_train=dt.predict(X_train)

accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)
print("Decsion Tree Classifier train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

# test accauracy
y_pred_test=dt.predict(X_test)

accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)

f1=f1_score(y_test,y_pred_test)
print("\nDecsion Tree Classifier test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

Decsion Tree Classifier train part analysis

accuracy 0.8031593894213702
pecision 0.651017214397496
recall 0.5565217391304348
f1 0.6000721240533718

Decsion Tree Classifier test part analysis

accuracy 0.7984386089425124
pecision 0.6347305389221557
recall 0.5668449197860963
f1 0.5988700564971752


In [62]:
# KNN with grid search
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn_with_grid = Pipeline(
    steps=[
        ("prepocessing", prepocessing),
        ("model", KNeighborsClassifier())
    ]
)


grid_param={
    "model__n_neighbors":[5,10,15,20,25],
    "model__weights":["uniform","distance"],
    "model__p":[1,2]
}


grid_search = GridSearchCV(
    estimator=knn_with_grid,
    param_grid=grid_param,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)


grid_search.fit(X_train, y_train)

#train accuracy part
y_pred_train = grid_search.predict(X_train)
accuracy=accuracy_score(y_train,y_pred_train)
pecision=precision_score(y_train,y_pred_train)
recall=recall_score(y_train,y_pred_train)
f1=f1_score(y_train,y_pred_train)

print("  KNN with gridsearch train part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)


# test accuracy
y_pred_test=dt.predict(X_test)
accuracy=accuracy_score(y_test,y_pred_test)
pecision=precision_score(y_test,y_pred_test)
recall=recall_score(y_test,y_pred_test)
f1=f1_score(y_test,y_pred_test)


print("\n KNN with gridsearch test part analysis\n")
print("accuracy", accuracy)
print("pecision", pecision)
print("recall", recall)
print("f1", f1)

  KNN with gridsearch train part analysis

accuracy 0.8130990415335463
pecision 0.671583850931677
recall 0.5785953177257525
f1 0.6216313330937837

 KNN with gridsearch test part analysis

accuracy 0.7984386089425124
pecision 0.6347305389221557
recall 0.5668449197860963
f1 0.5988700564971752


| Model                       |   Accuracy |  Precision |     Recall |   F1 Score | Overfitting |
| --------------------------- | ---------: | ---------: | ---------: | ---------: | ----------- |
| **Logistic Regression**     | **80.55%** |     65.72% |     55.88% | **60.40%** | ❌ No        |
| **Random Forest (Tuned)**   | **80.55%** | **67.01%** |     52.67% |     58.98% | ⚠️ Slight   |
| **Decision Tree (Untuned)** |     72.89% |     48.96% |     50.53% |     49.74% | ❌ Heavy     |
| **Decision Tree (Tuned)**   |     79.84% |     63.47% | **56.68%** |     59.89% | ✅ No        |
| **KNN + GridSearch**        |     79.84% |     63.47% | **56.68%** |     59.89% | ✅ No        |


After comparing all the classification models, I found that Logistic Regression performed the best on the Telco Customer Churn dataset. It achieved an accuracy of 80.55% and the highest F1 score of 60.4%. Another reason I selected this model is that its training and testing results were very close, which means it generalized well and did not overfit the data. Although Random Forest achieved the same accuracy and slightly better precision, its recall and F1 score were lower than Logistic Regression. The Decision Tree initially suffered from overfitting, but after hyperparameter tuning its performance improved significantly. Overall, considering accuracy, F1 score, and model stability, I believe Logistic Regression is the most suitable model for this dataset.

In [63]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Model
lr_cv=Pipeline(

    steps=[
        ("prepocessing", prepocessing),
        ("model", LogisticRegressionCV(
             Cs=[0.01, 0.1, 1, 2, 5, 10],
             cv=5,                          # 5-Fold Cross Validation
             class_weight="balanced",
             max_iter=6000,
             random_state=42))

    ]
)
# Train
lr_cv.fit(X_train, y_train)


#  Test
y_pred_test = lr_cv.predict(X_test)

print("\nTest Set")
print("Accuracy :", accuracy_score(y_test, y_pred_test))
print("Precision:", precision_score(y_test, y_pred_test, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred_test, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred_test, average="weighted"))


Test Set
Accuracy : 0.7423704755145494
Precision: 0.7991808094856337
Recall   : 0.7423704755145494
F1 Score : 0.7558611080511628
